Preprocessing

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

# --- 1. Konfigurasi Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# --- 2. Definisi Transformasi & Augmentasi Data ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),          # Ukuran standar ConvNeXt / ResNet
    transforms.RandomHorizontalFlip(),      # Augmentasi: Balik horizontal
    transforms.RandomRotation(30),          # Augmentasi: Rotasi
    transforms.ColorJitter(brightness=0.2), # Augmentasi: Kecerahan
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standar ImageNet
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 3. Memuat Dataset Menggunakan ImageFolder ---
train_dir = 'train'
val_dir = 'val'

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

try:
    val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transforms)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    print("Dataset validasi berhasil dimuat.")
except FileNotFoundError as e:
    print(f"Peringatan: Folder validasi tidak ditemukan atau kosong. Pastikan setiap sub-folder di '{val_dir}' memiliki minimal satu gambar.")
    val_dataset = None
    val_loader = None
except RuntimeError as e:
    print(f"Peringatan: Error saat memuat dataset validasi: {e}")
    val_dataset = None
    val_loader = None

# --- 4. Inisialisasi Model ConvNeXt ---
print("Memuat model ConvNeXt pre-trained...")
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

# --- 5. Memodifikasi Kepala Klasifikasi ---
nama_penyakit = train_dataset.classes
num_classes = len(nama_penyakit)
print(f"Berhasil mendeteksi {num_classes} kelas: {nama_penyakit}")

in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, num_classes)
model = model.to(device)

print("Model siap digunakan untuk training.")

Menggunakan device: cpu
Dataset validasi berhasil dimuat.
Memuat model ConvNeXt pre-trained...


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /Users/suniantara/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [01:05<00:00, 1.75MB/s] 


Berhasil mendeteksi 3 kelas: ['Demodex', 'Malassezia', 'Scabies']
Model siap digunakan untuk training.


Training

In [5]:
import torch.optim as optim

# --- 6. Menentukan Loss Function dan Optimizer ---
# CrossEntropyLoss adalah standar untuk klasifikasi multi-kelas (lebih dari 2 kelas)
criterion = nn.CrossEntropyLoss()

# Menggunakan optimizer AdamW yang sangat cocok untuk arsitektur modern seperti ConvNeXt
# Learning rate diset kecil (0.001) agar fine-tuning tidak merusak bobot pre-trained
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# --- 7. Siklus Pelatihan (Training Loop) ---
epochs = 10 # Jumlah iterasi model melihat seluruh dataset (bisa Anda ubah nanti)
print("\n=== Memulai Proses Training ===")

for epoch in range(epochs):
    # Set model ke mode training (mengaktifkan dropout & track batch normalization)
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        # Pindahkan data ke CPU/GPU
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Reset gradient agar tidak menumpuk dari iterasi sebelumnya
        optimizer.zero_grad() 

        # 2. Forward pass (Model menebak gambar)
        outputs = model(inputs) 
        
        # 3. Hitung error (Loss)
        loss = criterion(outputs, labels) 
        
        # 4. Backward pass (Menghitung gradient/arah perbaikan)
        loss.backward() 
        
        # 5. Update bobot model
        optimizer.step() 

        running_loss += loss.item()
        
        # Menghitung akurasi training batch ini
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Kalkulasi rata-rata loss dan akurasi per epoch
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    # --- 8. Evaluasi / Validasi (Jika folder val/ tersedia) ---
    val_loss = 0.0
    val_acc = 0.0
    
    if val_loader is not None:
        model.eval() # Set model ke mode evaluasi
        val_correct = 0
        val_total = 0
        
        with torch.no_grad(): # Matikan perhitungan gradient untuk menghemat memori
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    else:
        # Jika tidak ada validasi, cukup print hasil training
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

# --- 9. Menyimpan Model yang Sudah Dilatih ---
# Setelah selesai, simpan "otak" model agar bisa dipakai untuk memprediksi foto baru nanti
torch.save(model.state_dict(), 'model_kulit_anjing_convnext.pth')
print("\nTraining selesai! Bobot model berhasil disimpan di 'model_kulit_anjing_convnext.pth'.")


=== Memulai Proses Training ===
Epoch [1/10] | Train Loss: 0.5844, Train Acc: 65.03% | Val Loss: 0.4756, Val Acc: 66.67%
Epoch [2/10] | Train Loss: 0.5893, Train Acc: 65.36% | Val Loss: 0.4732, Val Acc: 66.67%
Epoch [3/10] | Train Loss: 0.5693, Train Acc: 64.38% | Val Loss: 0.4739, Val Acc: 66.67%
Epoch [4/10] | Train Loss: 0.5752, Train Acc: 64.05% | Val Loss: 0.4662, Val Acc: 66.67%
Epoch [5/10] | Train Loss: 0.5455, Train Acc: 68.30% | Val Loss: 0.4648, Val Acc: 66.67%
Epoch [6/10] | Train Loss: 0.5925, Train Acc: 66.01% | Val Loss: 0.4611, Val Acc: 100.00%
Epoch [7/10] | Train Loss: 0.5583, Train Acc: 68.95% | Val Loss: 0.4491, Val Acc: 66.67%
Epoch [8/10] | Train Loss: 0.5286, Train Acc: 69.28% | Val Loss: 0.4659, Val Acc: 66.67%
Epoch [9/10] | Train Loss: 0.5296, Train Acc: 71.24% | Val Loss: 0.4489, Val Acc: 66.67%
Epoch [10/10] | Train Loss: 0.5646, Train Acc: 69.61% | Val Loss: 0.4814, Val Acc: 66.67%

Training selesai! Bobot model berhasil disimpan di 'model_kulit_anjing_con

Testing

In [10]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Definisikan ulang arsitektur model (harus sama persis dengan saat training)
model = models.convnext_tiny(weights=None) # Tidak perlu download bobot imagenet lagi
num_classes = 3 # Demodex, Malassezia, Scabies
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, num_classes)

# 2. Muat bobot (otak) model yang sudah Anda simpan
model.load_state_dict(torch.load('model_kulit_anjing_convnext.pth', map_location=device))
model = model.to(device)
model.eval() # Set ke mode evaluasi (wajib)

# 3. Transformasi gambar (harus sama persis dengan val_transforms)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. Prediksi Gambar
image_path = 'gambar.jpeg' # Ganti dengan nama file foto Anda
image = Image.open(image_path).convert('RGB')
input_tensor = preprocess(image).unsqueeze(0).to(device) # Tambahkan dimensi batch

kelas_penyakit = ['Demodex', 'Malassezia', 'Scabies']

with torch.no_grad():
    outputs = model(input_tensor)
    probabilitas = torch.nn.functional.softmax(outputs[0], dim=0)
    _, predicted_idx = torch.max(outputs, 1)

    prediksi_final = kelas_penyakit[predicted_idx.item()]
    persentase = probabilitas[predicted_idx.item()].item() * 100

print(f"Hasil Prediksi: {prediksi_final} (Keyakinan: {persentase:.2f}%)")

/var/folders/q2/2h4p204s56d3vhv_jx5hmc6m0000gn/T/ipykernel_3393/2384569565.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('model_kulit

Hasil Prediksi: Malassezia (Keyakinan: 97.29%)
